# `ctc_loss`

These are my notes on CTCLoss. Some of this may be wrong because I was talking with LLMs to help me understand it.

According to the PyTorch documentation, [CTCLoss](https://docs.pytorch.org/docs/2.12/generated/torch.nn.CTCLoss.html) is a way of calculating the "loss between a continuous (unsegmented) time series and a target sequence."

But what it actually means by "continuous (unsegmented) time series" is really a discrete array, with an entry at each constant time step. For instance, let's say we have a recording of someone saying the word "hi". Perhaps the recording is one second long and we've broken it up into 0.1 second steps, so there are 10 steps in the time series. Each step contains some array of numbers. This is our "continuous (unsegmented) time series".

Let's say we have a model which takes the time series as its input, and we want it to detect what letter is being spoken at each time step in the recording. In other words, we want to create part of a speech-to-text model that can recognize the word "hi". The output of the model will be a sequence of vectors, where each vector encodes the probability of each character. For simplicity, let's say we only have three possible characters: 0. a blank character, 1. "H", and 2. "I". The output of the model will have shape `[10, 3]`, since we have 10 steps in the input series and 3 possible characters. Our target sequence will be `tensor([1, 2])`.

Initially the model is untrained, so it may output random garbage at each of the 10 steps, like:

In [1]:
import torch

torch.manual_seed(2)

def argmax_stringify(t):
    map = {
        0: '_',
        1: 'H',
        2: 'I',
    }
    return ''.join([map[idx] for idx in t.argmax(dim=-1).tolist()])

data_rand = torch.rand(10, 3)
data_rand /= data_rand.sum(dim=-1, keepdim=True)
print(argmax_stringify(data_rand))

target_sequence = torch.tensor([1, 2])
input_length = torch.tensor([10])
target_length = torch.tensor([2])

torch.ctc_loss(
    data_rand.log(),
    target_sequence,
    input_length,
    target_length,
)

IHIHHHHIHI


tensor(5.6830)

Let's say that in one of the recordings in our training data, a voice is saying "H" for the first 3 time steps and "I" for the last 7. After the model has gone through some iterations of training, its output would hopefully end up being closer to the ideal:

In [2]:
data_ideal = torch.tensor([
    [0., 1, 0], # --> H
    [0, 1, 0], # --> H
    [0, 1, 0], # --> H
    [0, 0, 1], # --> I
    [0, 0, 1], # --> I
    [0, 0, 1], # --> I
    [0, 0, 1], # --> I
    [0, 0, 1], # --> I
    [0, 0, 1], # --> I
    [0, 0, 1], # --> I
])

print(argmax_stringify(data_ideal))

torch.ctc_loss(
    data_ideal.log(),
    target_sequence,
    input_length,
    target_length,
)

HHHIIIIIII


tensor(-0.)

The CTC loss is 0 for this ideal model output. But an actual output may never quite reach the ideal, and instead it might be something like this while its part-way through training:

In [3]:
data_training = torch.tensor([
    [0.56, 0.30, 0.14], # --> blank
    [0.16, 0.73, 0.11], # --> H
    [0.07, 0.87, 0.06], # --> H
    [0.40, 0.25, 0.35], # --> blank
    [0.49, 0.10, 0.41], # --> blank
    [0.11, 0.02, 0.87], # --> I
    [0.06, 0.04, 0.90], # --> I
    [0.11, 0.09, 0.80], # --> I
    [0.11, 0.01, 0.88], # --> I
    [0.09, 0.02, 0.89], # --> I
    [0.48, 0.12, 0.40], # --> blank
])

print(argmax_stringify(data_training))

torch.ctc_loss(
    data_training.log(),
    target_sequence,
    input_length,
    target_length,
)

_HH__IIIII_


tensor(1.2318)

The CTC loss for this output is somewhere between that of the ideal output and the random one.

While the model is being trained on some labelled data, it will back propagate the CTC loss score to adjust the parameters of the model.

Once the model is trained and ready to use, the output would be processed by first removing repeated consecutive tokens and then removing blanks, like: `_HH__IIIII_ --> _H_I_ --> HI`.

However, argmax is not the only way to choose the tokens that we generate from the model's output probabilities. We could instead generate a weighted-random token for each entry using the probabilities of that entry as the weights.

In [4]:
def stringify_rand(t):
    indices = torch.multinomial(t, num_samples=1).flatten()
    map = {
        0: '_',
        1: 'H',
        2: 'I',
    }
    return ''.join([map[idx] for idx in indices.tolist()])

The ideal model output always produces the same result with this random sampling technique.

In [5]:
for _ in range(2):
    print(stringify_rand(data_ideal))

HHHIIIIIII
HHHIIIIIII


But a more realistic model output may produce different results each time.

In [6]:
for _ in range(2):
    print(stringify_rand(data_training))

H_H__IIIIII
HHHI_IIIII_


Given one set of model output probabilities (like `data_training`), one particular probabilistic sequence generated with it is called a path or trajectory. So `IIHIIII_III` is one path and `HIHH_IIIIII` is another path.

Many different paths can resolve to the same target label, which is `HI` in this example. For instance, `_HHH__II__` and `HH_IIIII__` would both resolve to `HI`.

We can find the probability of generating one particular path $\mathbf \pi$ from the model output probabilities $\mathbf y$ by just multiplying the probabilities at each time step for the corresponding token in the path. $T$ is the length of the path (also the number of probability vectors in the model output).

$$
P(\mathbf \pi) = \prod_{t=1}^T y^t_{\pi_t}
$$

$y^t_{\pi_t}$ is just the probability of token $\pi_t$ at time $t$, taken directly from the model output probabilities.

So for instance, if we want to find the probability of the path `HHHIIIIIII` (indices `1112222222`) from the model output `data_training`:

In [7]:
path = torch.tensor([1, 1, 1, 2, 2, 2, 2, 2, 2, 2])

print(data_training[torch.arange(10), path].prod())

tensor(0.0134)


The probability of that path is 0.5%. For the `data_ideal` output, it's 100%:

In [8]:
print(data_ideal[torch.arange(10), path].prod())

tensor(1.)


We can also calculate the probability of generating a path that resolves to our target label `HI`. If $\mathbf l$ is the target label and $x$ is the model input (in our case, the time series data representing the audio recording of someone saying "hi") which generates the output $y$:

$$
P(l | x) = \sum_{\pi \in B^{-1}(l)} P(\pi)
$$

$\pi \in B^-1(l)$ is just the set of all paths which collapse to the target label when we apply the collapsing rule of removing duplicate tokens and then removing blanks.

There are lots of ways to find all the paths that collapse to the target label, but we can use a brute force method for such a small example. Since there are 10 tokens with 3 possibilities, we have $3^10 \approx 60,000$ possible paths.

In [9]:
import itertools

def collapse_path(path):
    return [a for a, _ in itertools.groupby(path) if a != 0]

def path_to_string(path):
    path_collapsed = collapse_path(path)
    map = {
        0: "_",
        1: "H",
        2: "I",
    }
    return ''.join(map[idx] for idx in path_collapsed)

def gen_matching_paths(match=None):
    for path in itertools.product(range(3), repeat=10):
        if match is not None:
            c = collapse_path(path)
            if c == match:
                # Just double check that all matching paths are 'HI'
                assert path_to_string(path) == 'HI'
                yield path
        else:
            yield path

def sum_probs(model_output, paths):
    return model_output[torch.arange(10), paths].prod(dim=-1).sum()

all_paths = torch.tensor(list(gen_matching_paths()))
hi_paths = torch.tensor(list(gen_matching_paths([1, 2])))

print(f"Prob of all paths in data_rand={sum_probs(data_rand, all_paths):.4f}")
print(f"Prob of 'HI' paths in data_rand={sum_probs(data_rand, hi_paths):.4f}")
print('\n')
print(f"Prob of all paths in data_training={sum_probs(data_training, all_paths):.4f}")
print(f"Prob of 'HI' paths in data_training={sum_probs(data_training, hi_paths):.4f}")
print('\n')
print(f"Prob of all paths in data_ideal={sum_probs(data_ideal, all_paths):.4f}")
print(f"Prob of 'HI' paths in data_ideal={sum_probs(data_ideal, hi_paths):.4f}")


Prob of all paths in data_rand=1.0000
Prob of 'HI' paths in data_rand=0.0034


Prob of all paths in data_training=1.0000
Prob of 'HI' paths in data_training=0.2918


Prob of all paths in data_ideal=1.0000
Prob of 'HI' paths in data_ideal=1.0000


As we'd expect, for all three of our example model outpus, the probability of generating *any* path at all is 100%. The probability of generating a path that resolve to `HI` is the least for the random data, almost 0%, the most for the ideal data, exactly 100%, and the partially trained output is somewhere in the middle, at about 30%.

In [10]:
hi_paths.shape[0] / all_paths.shape[0]

0.008382868465172992

Only about 0.8% of the possible paths resolve to "HI", so the random output is in the right ballpark, and the partially trained output is much better than chance.

Now that we have a way to calculate the probability of generating the target label, the CTC loss is simply:

$$
L = -\log P(l | x)
$$

In [11]:

cases = [
    ('rand', data_rand),
    ('training', data_training),
    ('ideal', data_ideal),
]

for name, data in cases:
    print(f'\n{name}:')
    hi_prob_brute = sum_probs(data, hi_paths)
    ctc_loss_brute = -hi_prob_brute.log()
    ctc_loss_torch = torch.ctc_loss(
        data.log(),
        target_sequence,
        input_length,
        target_length,
    )
    hi_prob_torch = (-ctc_loss_torch).exp()

    print(f'  torch ctc_loss: {ctc_loss_torch:.4f}')
    print(f'  brute ctc_loss: {ctc_loss_brute:.4f}')
    print(f'  torch "HI" prob: {hi_prob_torch:.4f}')
    print(f'  brute "HI" prob: {hi_prob_brute:.4f}')



rand:
  torch ctc_loss: 5.6830
  brute ctc_loss: 5.6830
  torch "HI" prob: 0.0034
  brute "HI" prob: 0.0034

training:
  torch ctc_loss: 1.2318
  brute ctc_loss: 1.2318
  torch "HI" prob: 0.2918
  brute "HI" prob: 0.2918

ideal:
  torch ctc_loss: -0.0000
  brute ctc_loss: -0.0000
  torch "HI" prob: 1.0000
  brute "HI" prob: 1.0000


Above, we're comparing the results for our example brute-force CTC loss calculation to those of PyTorch, for all three of our example model outputs, and they match up well.